# Entrenamiento de modelos BiLSTM+CRF

Este notebook ejecuta múltiples experimentos de entrenamiento con y sin embeddings preentrenados para tareas de reconocimiento de entidades clínicas. Cada modelo se guarda en su propia carpeta y se evalúa automáticamente para comparar su desempeño.

In [ ]:
%pip install -q seqeval tf2crf "keras<3.0" tensorflow tensorflow-addons

: 

In [ ]:
from pathlib import Path
from pprint import pprint

import numpy as np

from config import (
    TRAIN_PATH, VALID_PATH, TEST_PATH, MAX_LEN, PAD_IDX,
    W2V_MODEL_PATH, BATCH_SIZE, EPOCHS
)
from data_utils import (
    load_bio_corpus, build_vocab, build_label_map,
    encode_sentences, encode_labels, get_idx2tag, get_idx2word
)
from model_utils import build_bilstm_crf_model, pad_sequences_tf
from train_eval_utils import (
    train_model, evaluate_model, save_model,
    plot_training_history
)
from embedding_utils import load_pretrained_embeddings


In [ ]:
# Cargar corpus
train_tokens, train_labels, _ = load_bio_corpus(TRAIN_PATH)
valid_tokens, valid_labels, _ = load_bio_corpus(VALID_PATH)
test_tokens, test_labels, _ = load_bio_corpus(TEST_PATH)

# Construcción de vocabulario
word2idx = build_vocab(train_tokens)
tag2idx = build_label_map(train_labels)
idx2tag = get_idx2tag(tag2idx)
idx2word = get_idx2word(word2idx)

# Codificación
X_train = encode_sentences(train_tokens, word2idx)
X_valid = encode_sentences(valid_tokens, word2idx)
X_test  = encode_sentences(test_tokens, word2idx)

y_train = encode_labels(train_labels, tag2idx)
y_valid = encode_labels(valid_labels, tag2idx)
y_test  = encode_labels(test_labels, tag2idx)

# Padding
X_train = pad_sequences_tf(X_train, maxlen=MAX_LEN, pad_value=PAD_IDX)
X_valid = pad_sequences_tf(X_valid, maxlen=MAX_LEN, pad_value=PAD_IDX)
X_test  = pad_sequences_tf(X_test, maxlen=MAX_LEN, pad_value=PAD_IDX)

y_train = pad_sequences_tf(y_train, maxlen=MAX_LEN, pad_value=PAD_IDX)
y_valid = pad_sequences_tf(y_valid, maxlen=MAX_LEN, pad_value=PAD_IDX)
y_test  = pad_sequences_tf(y_test, maxlen=MAX_LEN, pad_value=PAD_IDX)


In [ ]:
# Cargar matriz de embeddings (si existe)
embedding_matrix = load_pretrained_embeddings(W2V_MODEL_PATH, word2idx)

In [ ]:
experiments = [
    {
        "name": "baseline_enterizacion",
        "model_args": {
            "max_len": MAX_LEN,
            "n_words": len(word2idx),
            "n_tags": len(tag2idx),
            "embedding_matrix": None,
        },
    },
    {
        "name": "embedding_preentrenado",
        "model_args": {
            "max_len": MAX_LEN,
            "n_words": len(word2idx),
            "n_tags": len(tag2idx),
            "embedding_matrix": embedding_matrix,
        },
    },
]


In [ ]:
BASE_OUTPUT = Path("experiments")
BASE_OUTPUT.mkdir(exist_ok=True)

resultados = []

for exp in experiments:
    name = exp["name"]
    model_args = exp["model_args"]
    print(f"=== Entrenando modelo: {name} ===")

    model = build_bilstm_crf_model(**model_args)

    history = train_model(
        model,
        X_train,
        y_train,
        X_valid,
        y_valid,
        batch_size=BATCH_SIZE,
        epochs=EPOCHS,
    )

    plot_training_history(history, title=f"Curva de pérdida - {name}")

    _, _, f1 = evaluate_model(
        model, X_test, y_test, idx2tag=idx2tag, pad_value=PAD_IDX
    )

    exp_dir = BASE_OUTPUT / name
    if exp_dir.exists():
        shutil.rmtree(exp_dir)
    save_model(model, str(exp_dir))

    resultados.append({"modelo": name, "f1": round(f1, 4)})


In [ ]:
print("\n=== Resultados finales ===")
pprint(resultados)
